In [1]:
%pip install open3d mujoco pyyaml scipy matplotlib opencv-python


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# --- IMPORT REQUIRED LIBRARIES ---
from sim_fxn_lib import *
import csv
from matplotlib import cm
import cv2
import shutil
import numpy as np

In [3]:
# --- DEFINE SIMULATION SETTINGS + INITIALIZE MUJOCO MODEL ---
xml_path = 'RHex1.xml'
typ = "example"
#RFTCOEFF = 0.4
save_every = 8   
repeats = 3
tMax = 3
dt = 0.001
model, data, renderer, t, dt, frames, framerate, sand_h_id, stl_path = initialize_simulation(
    tMax,
    dt,
    xml_path, 
    'sandflipper.stl', 
    camera_name="plate_camera"
)
numSteps = len(t)
tMax = t[-1]

box_id = mujoco.mj_name2id(
    model,
    mujoco.mjtObj.mjOBJ_BODY,
    "box"
)

In [4]:
# --- MAPPING NAMES TO ID'S ---
pos_actuator_ids = {}
actuator_names = [
    "front right_p", "front left_p",
    "middle right_p", "middle left_p",
    "back right_p", "back left_p"
]
for name in actuator_names:
    pos_actuator_ids[name] = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, name)
sand_h_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "sand_height")

In [5]:
# --- IMPORTING + USING STL MESH GEOMETRIES ---
entities = get_named_bodies_from_xml(xml_path)

body, vertices, faces, mesh = load_and_process_mesh(stl_path, scale_factor=1000)
num_sites = model.nsite - 7
num_faces = len(mesh.triangles)
num_active_sites = min(num_sites, num_faces)

vertices_dict = {}
faces_dict = {}
body_dict = {}
mesh_dict = {}

for body_name in entities:
    stl_path = f"asset/{body_name}.stl"
    body, vertices, faces, mesh = load_and_process_mesh(stl_path, scale_factor=1000)
    vertices_dict[body_name] = vertices
    faces_dict[body_name] = faces
    body_dict[body_name] = body
    mesh_dict[body_name] = mesh
    
    body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body_name)

[Open3D WARNING] Unable to load file asset/box.stl with ASSIMP: Unable to open file "asset/box.stl".


In [6]:
# --- APPLYING FORCE SITES ON MESH ---   
def initialize_sites_for_all_bodies(model, data, body_dict, sitename="force"):
    for body_name, mesh in mesh_dict.items():
        initialize_sites_on_mesh(
            model=model,
            data=data,
            mesh=mesh,
            sitename=f"{sitename}_{body_name}",
            bodyname=body_name
        )
        
initialize_sites_for_all_bodies(model, data, body_dict, sitename="force")

In [7]:
# --- DATA --- 
body_list = entities

v = {}
for entity in entities:
    v[f'fm_{entity}'] = []
    v[f'mm_{entity}'] = []
    
motion_data = {
    "time": [],
    "x": [],
    "y": [],
    "z": [],
    "roll": [],
    "pitch": [],
    "yaw": []
}

camera_list = ["diag"]
frames = {cam: [] for cam in camera_list}
plate_pos = []

In [8]:
# --- STORING PREVIOUS POSITION + ORIENTATION TO ESTIMATE VELOCITY ---
F_full_sorted_prev = np.zeros_like(faces)
prev_body_pos_dict = {name: None for name in entities}
prev_body_quat_dict = {name: None for name in entities}

In [9]:
# --- SET UP STUFF ---
applied_force = True
once_submerged = False
frame_dir = "frames"
os.makedirs(frame_dir, exist_ok=True)
csv_folder = "csv"
os.makedirs(csv_folder, exist_ok=True)
for filename in os.listdir(frame_dir):
    file_path = os.path.join(frame_dir, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print('Failed to delete %s. Reason: %s' % (file_path, e))

In [10]:
# --- MAIN SIM ---
body_velocities = {body_name: [] for body_name in entities}
body_angular_velocities = {body_name: [] for body_name in entities}
site_ids = {}
SUB = False
for body_name in body_list:
    site_ids[f"{body_name}"] = [
        mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, f"force_{body_name}_site_{s}")
        for s in range(num_sites)
    ]
site_id_arrays = {body_name: np.array(ids) for body_name, ids in site_ids.items()}
print(body_list)
face_sort_order = {}
sorted_faces_cache = {}
sorted_site_ids_cache = {}

['plate', 'box', 'mid right', 'mid left', 'front right', 'front left', 'back right', 'back left']


In [11]:
# - SORTING MESH X-COORDINATE AND FORCE - 
for body_name in body_list:
    faces, verts = faces_dict[body_name], vertices_dict[body_name]
    centroids_x = np.mean(verts[faces], axis=1)[:, 0]
    order = np.argsort(centroids_x)
    face_sort_order[body_name]       = order
    sorted_faces_cache[body_name]    = faces[order]
    sorted_site_ids_cache[body_name] = np.array(site_ids[body_name])[order]
    
# - AREA OF MESH TRIANGLES - 
face_areas_cache = {
    body_name: calculate_face_areas(vertices, sorted_faces_cache[body_name])
    for body_name in body_list
}

The following blocks are code for gait control

In [12]:
# set up
plate_id = mujoco.mj_name2id(model,mujoco.mjtObj.mjOBJ_BODY,"plate")
start_pos = data.xpos[plate_id].copy()

# -------- LEFT TRIPOD GAIT ---------
tripod_left = ["mid right" , "front left" , "back left"]
joint_id_mr = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_left[0])
joint_id_fl = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_left[1])
joint_id_bl = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_left[2])

# -------- RIGHT TRIPOD GAIT ---------
tripod_right = ["mid left" , "front right" , "back right"]
joint_id_ml = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_right[0])
joint_id_fr = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_right[1])
joint_id_br = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, tripod_right[2])

# starting angles
commanded_left_angle = 0.0 
commanded_right_angle = -160*(np.pi/180)

In [13]:
def create_video_from_frames(typ, frame_folder, frame_prefix, save_every, output_name=None):
    fps = 1000 / save_every

    if output_name is None:
        output_video = f"{typ}_rigid.mp4"
    else:
        output_video = output_name

    frames = sorted([
        f for f in os.listdir(frame_folder)
        if f.startswith(frame_prefix) and f.endswith(".png")
    ])

    if len(frames) == 0:
        return None

    first_frame = cv2.imread(os.path.join(frame_folder, frames[0]))
    height, width, _ = first_frame.shape

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    for f in frames:
        img = cv2.imread(os.path.join(frame_folder, f))

        if img is None:
            continue

        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

        out.write(img)

    out.release()
    print(f"Video saved to {output_video}")

    return output_video

In [14]:
left_leg_state = "swing"
right_leg_state = "swing"
mr_touching_sand = False
ml_touching_sand = False

#using 5 kgs as standard payload, estimated ara deviced is ~ 10lbs
payload_list = [5]
                #3,5,7,9,11,13,15,17,19,21,23,25]
RFTCOEFF_list = [0.07,0.10,0.20,0.30,0.40,0.50,0.75,1.00,1.50,2.00,3.00,4.00,10.00]
#[0.075, 0.1 , 0.2 , 0.3 , 0.4 , 0.5 , 0.75 , 1.0 , 1.5 , 2 , 2.5 , 3 , 3.5 , 4 , 4.5 , 5 , 6 , 8 , 10] 

stance_list = [0.16]
    #0.05714,0.07428,0.09143,0.108057,0.12571,0.14286,0.16,0.17714,0.19429,0.21143,0.22857,0.24571]
swing_list = [0.56]
    #0.2,0.26,0.32,0.38,0.44,0.50,0.56,0.62,0.68,0.74,0.80,0.86]

start_list = [180,170,160,150,140,130,120,110,100]
end_list = [260,250,240,230,220,210,200,190,180]

# to change payload aka box mass from xml ... 
og_mass = model.body_mass[box_id].copy()
og_inertia = model.body_inertia[box_id].copy()

for payload in payload_list:
    model.body_mass[box_id] = payload

    model.body_inertia[box_id] = (og_inertia * payload / og_mass)

    mujoco.mj_setConst(model, data)

    mujoco.mj_resetData(model, data)
    mujoco.mj_forward(model, data)

    
    for rft_coeff in RFTCOEFF_list:
        for a, b in zip(stance_list, swing_list):
            stance_step = a * (np.pi / 180)
            swing_step = b * (np.pi / 180)

            for c, d in zip(start_list, end_list):

                # reset simulation for each parameter set
                mujoco.mj_resetData(model, data)
                mujoco.mj_forward(model, data)

                commanded_left_angle = 0
                commanded_right_angle = -160*(np.pi/180)

                start_pos = data.xpos[plate_id].copy()

                plate_pos = []

                once_submerged = False

                prev_body_pos_dict = {}
                prev_body_quat_dict = {}

                body_velocities = {}
                body_angular_velocities = {}

                for body_name in body_list:
                    prev_body_pos_dict[body_name] = None
                    prev_body_quat_dict[body_name] = None
                    body_velocities[body_name] = []
                    body_angular_velocities[body_name] = []

                start_stance_list = []
                end_stance_list = []

                stance_reps = 20
                for cycle in range(stance_reps):
                    start_stance = c * (np.pi / 180) - 2 * np.pi * cycle
                    end_stance = d * (np.pi / 180) - 2 * np.pi * cycle

                    start_stance_list.append(start_stance)
                    end_stance_list.append(end_stance)

                print("Running test:")
                print("stance step:", a)
                print("swing step:", b)
                print("start angle:", c)
                print("end angle:", d)
                print("sand stiffness:", rft_coeff)
                print("payload:", payload)

                for i in range(len(t)):

                    left_leg_state = "swing"
                    right_leg_state = "swing"

                    for j in range(len(start_stance_list)):
                        start_stance = start_stance_list[j]
                        end_stance = end_stance_list[j]

                        if commanded_left_angle >= start_stance and commanded_left_angle < end_stance:
                            left_leg_state = "stance"

                        if commanded_right_angle >= start_stance and commanded_right_angle < end_stance:
                            right_leg_state = "stance"

                    if left_leg_state == "stance":
                        commanded_left_angle -= stance_step
                    else:
                        commanded_left_angle -= swing_step

                    if right_leg_state == "stance":
                        commanded_right_angle -= stance_step
                    else:
                        commanded_right_angle -= swing_step

                    data.ctrl[pos_actuator_ids["front left_p"]] = commanded_left_angle
                    data.ctrl[pos_actuator_ids["middle right_p"]] = commanded_left_angle
                    data.ctrl[pos_actuator_ids["back left_p"]] = commanded_left_angle

                    data.ctrl[pos_actuator_ids["front right_p"]] = commanded_right_angle
                    data.ctrl[pos_actuator_ids["middle left_p"]] = commanded_right_angle
                    data.ctrl[pos_actuator_ids["back right_p"]] = commanded_right_angle

                    mujoco.mj_step(model, data)

                    data.qfrc_applied[:] = 0

                    global_pos_sand = data.site_xpos[sand_h_id]

                    robot_pos = data.xpos[plate_id].copy()
                    dist_x = robot_pos[0] - start_pos[0]

                    mr_touching_sand = False
                    ml_touching_sand = False

                    for body_name in body_list:

                        body_id = mujoco.mj_name2id(
                            model,
                            mujoco.mjtObj.mjOBJ_BODY,
                            body_name
                        )

                        if body_name == "plate":
                            plate_pos.append(np.array(data.xpos[body_id]))

                        ids_arr = site_id_arrays[body_name]
                        site_z = data.site_xpos[ids_arr, 2]
                        SUB = bool(np.any(site_z < global_pos_sand[2]))

                        if body_name == "middle right":
                            mr_touching_sand = SUB
                        elif body_name == "middle left":
                            ml_touching_sand = SUB

                        body_pos = np.array(data.xpos[body_id])
                        quat_now = np.array(data.xquat[body_id])

                        if i > 0 and prev_body_pos_dict[body_name] is not None:
                            prev_body_pos = prev_body_pos_dict[body_name]
                            prev_body_quat = prev_body_quat_dict[body_name]

                            body_linear_velocity = (body_pos - prev_body_pos) / dt

                            quat_prev = np.array(prev_body_quat)

                            r_now = scipy.spatial.transform.Rotation.from_quat([
                                quat_now[1],
                                quat_now[2],
                                quat_now[3],
                                quat_now[0]
                            ])

                            r_prev = scipy.spatial.transform.Rotation.from_quat([
                                quat_prev[1],
                                quat_prev[2],
                                quat_prev[3],
                                quat_prev[0]
                            ])

                            rotvec = (r_now * r_prev.inv()).as_rotvec()
                            body_angular_velocity = rotvec / dt

                        else:
                            body_linear_velocity = np.zeros(3)
                            body_angular_velocity = np.zeros(3)

                        body_velocities[body_name].append(body_linear_velocity.copy())
                        body_angular_velocities[body_name].append(body_angular_velocity.copy())

                        prev_body_pos_dict[body_name] = body_pos.copy()
                        prev_body_quat_dict[body_name] = quat_now.copy()

                        body_orientation_world = data.xquat[body_id]
                        euler_angles = quaternion_to_euler(body_orientation_world)
                        roll, pitch, yaw = euler_angles

                        F_muj = np.array([0, 0, 0])
                        M_muj = np.array([0, 0, 0])

                        if SUB:
                            if once_submerged is False:
                                once_submerged = True

                            F_muj, M_muj, Fi_mat, Mi_mat, include, F_full, rft_coeff = rft_3D_body_full_mat(
                                body_dict[body_name],
                                body_pos,
                                [yaw, pitch, roll],
                                body_linear_velocity,
                                body_angular_velocity,
                                RFTCOEFF=rft_coeff,
                                sand_height_m=global_pos_sand[2]
                            )

                            order = face_sort_order[body_name]
                            faces_sorted = sorted_faces_cache[body_name]
                            site_ids_sorted = sorted_site_ids_cache[body_name]

                            F_full_sorted = F_full[order]
                            face_areas = face_areas_cache[body_name]

                            stress_plot = F_full_sorted / face_areas[:, np.newaxis]
                            stress_magnitude = np.linalg.norm(stress_plot, axis=1)

                            if stress_magnitude.size > 0 and stress_magnitude.max() != stress_magnitude.min():
                                norm_stress = (
                                    stress_magnitude - stress_magnitude.min()
                                ) / (
                                    stress_magnitude.max() - stress_magnitude.min()
                                )
                            else:
                                norm_stress = np.zeros_like(stress_magnitude)

                            face_colors = cm.Blues(norm_stress)[:, :3]

                            if "prev_forces" not in globals():
                                prev_forces = {}

                            for face_idx, site_id in enumerate(site_ids_sorted):

                                if face_idx < len(F_full_sorted) and applied_force is True:
                                    force_applied_at_site = np.array(
                                        -F_full_sorted[face_idx],
                                        dtype=np.float64
                                    )

                                    alpha = 0.5

                                    if site_id in prev_forces:
                                        force_applied_at_site = (
                                            alpha * force_applied_at_site
                                            + (1 - alpha) * prev_forces[site_id]
                                        )

                                    prev_forces[site_id] = force_applied_at_site

                                    force_applied_at_site = force_applied_at_site.reshape((3, 1))

                                    mujoco.mj_applyFT(
                                        model,
                                        data,
                                        force_applied_at_site,
                                        np.zeros((3, 1), dtype=np.float64),
                                        np.array(data.site_xpos[site_id]).reshape((3, 1)),
                                        body_id,
                                        data.qfrc_applied
                                    )

                            v[f"fm_{body_name}"].append(F_muj)
                            v[f"mm_{body_name}"].append(M_muj)

                        else:
                            v[f"fm_{body_name}"].append(np.array([0, 0, 0]))
                            v[f"mm_{body_name}"].append(np.array([0, 0, 0]))
                    if i % save_every == 0:

                        renderer.update_scene(data, camera="diag")
                        frame = renderer.render()

                        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

                        cv2.imwrite(
                            f"frames/{typ}_start_{c:.2f}_rft_{rft_coeff:.2f}_frame_{i:04d}.png",
                            frame
                        )

                print("final distance traveled:", dist_x)

                csv_name = os.path.join(
                    csv_folder,
                    f"{typ}_plate_position_start_{c:.2f}_rft_{rft_coeff:.2f}"
                    f".csv"
                )

                with open(csv_name, "w", newline="") as f:
                    writer = csv.writer(f)

                    writer.writerow([
                        "X [m]",
                        "Y [m]",
                        "Z [m]"
                    ])

                    for pos in plate_pos:
                        row = [f"{coord:.4f}" for coord in pos]
                        writer.writerow(row)

                print("CSV saved to:", csv_name)

                create_video_from_frames(
                    typ=typ,
                    frame_folder="frames",
                    frame_prefix=f"{typ}_start_{c:.2f}_rft_{rft_coeff:.2f}_frame_",
                    save_every=save_every,
                    output_name=f"videos/{typ}_start_{c:.2f}_rft_{rft_coeff:.2f}.mp4"
                )


Running test:
stance step: 0.16
swing step: 0.56
start angle: 180
end angle: 260
sand stiffness: 0.07
payload: 5


/var/folders/7l/gqfqdk_n6bb9sqw5qgx6x22w0000gn/T/ipykernel_46876/579162924.py:217: RuntimeWarning: divide by zero encountered in divide
  stress_plot = F_full_sorted / face_areas[:, np.newaxis]
/var/folders/7l/gqfqdk_n6bb9sqw5qgx6x22w0000gn/T/ipykernel_46876/579162924.py:217: RuntimeWarning: invalid value encountered in divide
  stress_plot = F_full_sorted / face_areas[:, np.newaxis]


final distance traveled: -0.005316301008078658
CSV saved to: csv/example_plate_position_start_180.00_rft_0.07.csv
Video saved to videos/example_start_180.00_rft_0.07.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 170
end angle: 250
sand stiffness: 0.07
payload: 5
final distance traveled: -0.0004812115383186923
CSV saved to: csv/example_plate_position_start_170.00_rft_0.07.csv
Video saved to videos/example_start_170.00_rft_0.07.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 160
end angle: 240
sand stiffness: 0.07
payload: 5
final distance traveled: 0.002989999561838294
CSV saved to: csv/example_plate_position_start_160.00_rft_0.07.csv
Video saved to videos/example_start_160.00_rft_0.07.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 150
end angle: 230
sand stiffness: 0.07
payload: 5
final distance traveled: 0.004667244136934439
CSV saved to: csv/example_plate_position_start_150.00_rft_0.07.csv
Video saved to videos/example_start_15

Video saved to videos/example_start_160.00_rft_0.30.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 150
end angle: 230
sand stiffness: 0.3
payload: 5
final distance traveled: 0.3630266397618634
CSV saved to: csv/example_plate_position_start_150.00_rft_0.30.csv
Video saved to videos/example_start_150.00_rft_0.30.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 140
end angle: 220
sand stiffness: 0.3
payload: 5
final distance traveled: 0.34106708223410726
CSV saved to: csv/example_plate_position_start_140.00_rft_0.30.csv
Video saved to videos/example_start_140.00_rft_0.30.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 130
end angle: 210
sand stiffness: 0.3
payload: 5
final distance traveled: 0.31509464690288685
CSV saved to: csv/example_plate_position_start_130.00_rft_0.30.csv
Video saved to videos/example_start_130.00_rft_0.30.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 120
end angle: 200
sand stiffness: 0.3
payl

final distance traveled: 0.5480224735562792
CSV saved to: csv/example_plate_position_start_130.00_rft_0.75.csv
Video saved to videos/example_start_130.00_rft_0.75.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 120
end angle: 200
sand stiffness: 0.75
payload: 5
final distance traveled: 0.5183720191987499
CSV saved to: csv/example_plate_position_start_120.00_rft_0.75.csv
Video saved to videos/example_start_120.00_rft_0.75.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 110
end angle: 190
sand stiffness: 0.75
payload: 5
final distance traveled: 0.5115163038272833
CSV saved to: csv/example_plate_position_start_110.00_rft_0.75.csv
Video saved to videos/example_start_110.00_rft_0.75.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 100
end angle: 180
sand stiffness: 0.75
payload: 5
final distance traveled: 0.511025577132407
CSV saved to: csv/example_plate_position_start_100.00_rft_0.75.csv
Video saved to videos/example_start_100.00_rft_0.7

Video saved to videos/example_start_110.00_rft_2.00.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 100
end angle: 180
sand stiffness: 2.0
payload: 5
final distance traveled: 0.7460002810304408
CSV saved to: csv/example_plate_position_start_100.00_rft_2.00.csv
Video saved to videos/example_start_100.00_rft_2.00.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 180
end angle: 260
sand stiffness: 3.0
payload: 5
final distance traveled: 0.6986320590929077
CSV saved to: csv/example_plate_position_start_180.00_rft_3.00.csv
Video saved to videos/example_start_180.00_rft_3.00.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 170
end angle: 250
sand stiffness: 3.0
payload: 5
final distance traveled: 0.6570645532292626
CSV saved to: csv/example_plate_position_start_170.00_rft_3.00.csv
Video saved to videos/example_start_170.00_rft_3.00.mp4
Running test:
stance step: 0.16
swing step: 0.56
start angle: 160
end angle: 240
sand stiffness: 3.0
payloa